In [0]:
%sql
-- cretae silver table
CREATE TABLE IF NOT EXISTS silver.day18_customers (
    CustomerId INT,
    CustomerName STRING,
    City STRING,
    Age INT,
    UpdatedAt TIMESTAMP
)
USING DELTA;
-- create quarantine table
CREATE TABLE IF NOT EXISTS silver.day18_customer_quarantine (
    CustomerId INT,
    CustomerName STRING,
    City STRING,
    Age INT,
    UpdatedAt TIMESTAMP,
    ErrorReason STRING,
    RejectedAt TIMESTAMP
)
USING DELTA;

In [0]:
from pyspark.sql.types import (
    StructType,
    StructField,
    IntegerType,
    StringType,
    TimestampType
)

customer_schema = StructType([
    StructField("CustomerId", IntegerType(), True),
    StructField("CustomerName", StringType(), True),
    StructField("City", StringType(), True),
    StructField("Age", IntegerType(), True),
    StructField("UpdatedAt", TimestampType(), True)
])
bronze_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option("schema", customer_schema)
    .option("cloudFiles.schemaLocation", "/Volumes/workspace/bronze/schema/day18_customers/")
    .load("/Volumes/workspace/bronze/day18_customer_files/")
)


In [0]:
from pyspark.sql import functions as F

def process_batch(batch_df, batch_id):
    #write to bronze
    batch_df.write.mode("append").saveAsTable("bronze.day18_customers")
    #detect duplicate customerId
    duplicate_ids = (
    batch_df.groupBy("CustomerId")
      .count()
      .filter(F.col("count") > 1)
        )
    #add dupilcate flag
    df_with_duplicate_flag = (
    batch_df.join(
        duplicate_ids.select("CustomerId").withColumn("IsDuplicate", F.lit(True)),
        on="CustomerId",
        how="left"
    )
    .fillna({"IsDuplicate": False})
    )
    #create validation rule
    #rule 1: customer_id cannot be null
    customer_id_error = F.col("CustomerId").isNull()
    #rule 2: CustomerName cannot be null or empty
    customer_name_error = (
    F.col("CustomerName").isNull() |
    (F.trim(F.col("CustomerName")) == "")
    )
    #rule 3: City cannot be null or empty
    city_error = (
    F.col("City").isNull() |
    (F.trim(F.col("City")) == "")
    ) 
    #rule 4: Age cannot be null or less than 0 or greater than 100
    age_error = (
    F.col("Age").isNull() |
    (F.col("Age") < 0) |
    (F.col("Age") > 100)
    )
    #rule 5: updatedAt cannot be null
    updated_at_error = F.col("UpdatedAt").isNull()
    #rule 6: duplicate check
    duplicate_error = F.col("IsDuplicate") == True
    #Error_Reason 
    validated_df = (
        df_with_duplicate_flag
        .withColumn(
            "ErrorReason",
            F.concat_ws(
                "; ",
                F.when(customer_id_error, "CustomerId is NULL"),
                F.when(customer_name_error, "CustomerName is NULL"),
                F.when(city_error, "City is NULL"),
                F.when(age_error, "Age is outside valid range"),
                F.when(updated_at_error, "UpdatedAt is NULL"),
                F.when(duplicate_error, "Duplicate CustomerId")
            )
        )
    )
    #valid data
    valid_df = (
    validated_df
    .filter(
        F.col("ErrorReason").isNull() |
        (F.col("ErrorReason") == "")
    )
    .drop("IsDuplicate", "ErrorReason","_rescued_data")
    )   
    #invalid data
    invalid_df = (
    validated_df
    .filter(
        F.col("ErrorReason").isNotNull() &
        (F.col("ErrorReason") != "")
     )
    .withColumn("RejectedAt",F.current_timestamp())
    .drop("IsDuplicate", "_rescued_data")
    ) 
    #write to silver table
    valid_df.write.mode("append").saveAsTable("silver.day18_customers")
    #write to quarantine table
    invalid_df.write.mode("append").saveAsTable("silver.day18_customer_quarantine")
    #data_quality_check
    valid_count = valid_df.count()
    invalid_count = invalid_df.count()
    print(f"Valid records   : {valid_count}")
    print(f"Invalid records : {invalid_count}")
    total_count = valid_count + invalid_count
    quality_percentage = (
    valid_count / total_count * 100
    )
    print(f"Data Quality : {quality_percentage:.2f}%")

query = (
    bronze_stream.writeStream
    .foreachBatch(process_batch)
    .option("checkpointLocation", "/Volumes/workspace/bronze/checkpoints/day18_customer_data/")
    .trigger(availableNow=True)
    .start()
)
query.awaitTermination()

In [0]:
%sql
select * from bronze.day18_customers;

CustomerId,CustomerName,City,Age,UpdatedAt,_rescued_data
105,Ravi,Chennai,28,2026-08-30 09:00:00,null
106,null,Bangalore,35,2026-08-30 09:01:00,null
107,Arun,Chennai,-5,2026-08-30 09:02:00,null
108,Priya,Chennai,150,2026-08-30 09:03:00,null
108,Priya,Chennai,30,2026-08-30 09:04:00,null
109,Meena,Madurai,29,2026-08-30 09:05:00,null
110,Suresh,Chennai,32,2026-08-30 10:00:00,null
111,Anitha,Coimbatore,26,2026-08-30 10:01:00,null
112,John,Bangalore,40,2026-08-30 10:02:00,null


In [0]:
%sql
select * from silver.day18_customers;

CustomerId,CustomerName,City,Age,UpdatedAt
110,Suresh,Chennai,32,2026-08-30T10:00:00.000Z
111,Anitha,Coimbatore,26,2026-08-30T10:01:00.000Z
112,John,Bangalore,40,2026-08-30T10:02:00.000Z
105,Ravi,Chennai,28,2026-08-30T09:00:00.000Z
109,Meena,Madurai,29,2026-08-30T09:05:00.000Z


In [0]:
%sql
select * from silver.day18_customer_quarantine;

CustomerId,CustomerName,City,Age,UpdatedAt,ErrorReason,RejectedAt
106,null,Bangalore,35,2026-08-30T09:01:00.000Z,CustomerName is NULL,2026-08-30T08:35:24.109Z
107,Arun,Chennai,-5,2026-08-30T09:02:00.000Z,Age is outside valid range,2026-08-30T08:35:24.109Z
108,Priya,Chennai,150,2026-08-30T09:03:00.000Z,Age is outside valid range; Duplicate CustomerId,2026-08-30T08:35:24.109Z
108,Priya,Chennai,30,2026-08-30T09:04:00.000Z,Duplicate CustomerId,2026-08-30T08:35:24.109Z
